# vamprhyme-llm: fine-tuning on rhyme pairs

Runtime > Change runtime type > **T4 GPU**. See `pairs.md` for the full workflow.

## 1. Check the GPU and install libraries

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers peft
!pip uninstall -y -q torchao
import torch
print("CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUNS = "/content/drive/MyDrive/vamprhyme/runs"

## 3. Get the project

In [ ]:
import os
from google.colab import userdata

REPO = "YOUR_GITHUB_USERNAME/vamprhyme-llm"
PRIVATE = True

if PRIVATE:
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
else:
    url = f"https://github.com/{REPO}.git"

if os.path.isdir("/content/vamprhyme-llm"):
    !git -C /content/vamprhyme-llm pull
else:
    !git clone -q {url} /content/vamprhyme-llm

%cd /content/vamprhyme-llm

## 4. Copy the pair data from Drive

In [ ]:
!unzip -oq /content/drive/MyDrive/vamprhyme/vamprhyme-sft-pairs.zip -d /content/vamprhyme-llm
!wc -l data/sft-pairs/*.jsonl

## 5. Fine-tune on pairs and style examples

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python finetune_lora.py --data data/sft-pairs --epochs 2 --out {RUNS}/lora-pairs

## 6. Test: base model vs pair-trained adapter

In [ ]:
!python chat.py --adapter {RUNS}/lora-pairs/adapter --compare --max-new-tokens 200 \
    --prompts-file data/sft-pairs/test_prompts.jsonl

## 7. The Step 10 adapter on the same prompts

In [ ]:
!python chat.py --adapter {RUNS}/lora-styles/adapter --max-new-tokens 200 \
    --prompts-file data/sft-pairs/test_prompts.jsonl

## 8. Loss curves

In [ ]:
from IPython.display import Image
Image(f"{RUNS}/lora-pairs/loss.png")